In [1]:
import pandas as pd
import numpy as np
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt
import seaborn as sns
import time


In [ ]:
def read_mnist_images(path):
    """Read MNIST image binary file"""
    with open(path, 'rb') as f:
        magic = np.frombuffer(f.read(4), dtype='>i4')[0]
        n = np.frombuffer(f.read(4), dtype='>i4')[0]
        rows = np.frombuffer(f.read(4), dtype='>i4')[0]
        cols = np.frombuffer(f.read(4), dtype='>i4')[0]
        images = np.frombuffer(f.read(), np.uint8).reshape(n, rows, cols)
    return images

def read_mnist_labels(path):
    """Read MNIST label binary file"""
    with open(path, 'rb') as f:
        magic = np.frombuffer(f.read(4), dtype='>i4')[0]
        n = np.frombuffer(f.read(4), dtype='>i4')[0]
        labels = np.frombuffer(f.read(), np.uint8)
    return labels

In [ ]:
# Load training data
train_images = read_mnist_images('./Dataset/MNIST/train-images.idx3-ubyte')
train_labels = read_mnist_labels('./Dataset/MNIST/train-labels.idx1-ubyte')

# Load test data
test_images = read_mnist_images('./Dataset/MNIST/t10k-images.idx3-ubyte')
test_labels = read_mnist_labels('./Dataset/MNIST/t10k-labels.idx1-ubyte')

print(f"Train images shape: {train_images.shape}")  # (60000, 28, 28)
print(f"Train labels shape: {train_labels.shape}")  # (60000,)
print(f"Test images shape: {test_images.shape}")    # (10000, 28, 28)
print(f"Test labels shape: {test_labels.shape}")  

In [ ]:
train_images = train_images / 255.0
test_images = test_images / 255.0

In [ ]:
train_images = train_images.reshape(-1, 28*28)
test_images = test_images.reshape(-1, 28*28)

In [ ]:
train_images.shape

In [ ]:
test_images.shape  # (60000, 784)

In [ ]:
print(train_images[0].min(), train_images[0].max())

In [ ]:
X = train_images
y = train_labels

In [ ]:

# Ensure data is 2D before training
X_reshaped = X.reshape(X.shape[0], -1)
test_images_reshaped = test_images.reshape(test_images.shape[0], -1)

# ── Training ──────────────────────────────────────────────
train_start = time.time()

model = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=20, random_state=42)
model.fit(X_reshaped, y)

train_end = time.time()
training_time = train_end - train_start

# ── Inference (full test set) ─────────────────────────────
infer_start = time.time()

y_pred = model.predict(test_images_reshaped)

infer_end = time.time()
inference_time = infer_end - infer_start

# ── Inference (single image) ──────────────────────────────
single_start = time.time()

model.predict(test_images_reshaped[:1])

single_end = time.time()
single_image_time = single_end - single_start

# ── Metrics ───────────────────────────────────────────────
accuracy  = accuracy_score(test_labels, y_pred)
precision = precision_score(test_labels, y_pred, average='weighted')
recall    = recall_score(test_labels, y_pred, average='weighted')
f1        = f1_score(test_labels, y_pred, average='weighted')

# ── Results ───────────────────────────────────────────────
print("=" * 40)
print("         PERFORMANCE METRICS")
print("=" * 40)
print(f"Accuracy:   {accuracy:.4f}")
print(f"Precision:  {precision:.4f}")
print(f"Recall:     {recall:.4f}")
print(f"F1-Score:   {f1:.4f}")
print("-" * 40)
print("         TIMING")
print("-" * 40)
print(f"Training time:            {training_time:.4f} s")
print(f"Inference (10,000 imgs):  {inference_time:.4f} s")
print(f"Inference (1 image):      {single_image_time * 1000:.4f} ms")
print(f"Avg per image:            {(inference_time / len(test_images_reshaped)) * 1000:.4f} ms")
print("=" * 40)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Confusion Matrix
cm = confusion_matrix(test_labels, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - MNIST Test Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

# 2. Classification Report
print("\nClassification Report:")
print(classification_report(test_labels, y_pred))

In [ ]:
train_pred = model.predict(X_reshaped)
train_accuracy = accuracy_score(y, train_pred)

print(f"\n=== Model Performance Comparison ===")
print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy:     {accuracy:.4f}")
print(f"Difference:        {abs(train_accuracy - accuracy):.4f}")

In [ ]:
print("\n=== K-Fold Cross-Validation ===")
cv_scores = cross_val_score(MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=20, random_state=42), 
                             X_reshaped, y, cv=5, scoring='accuracy')

print(f"CV Fold Scores: {cv_scores}")
print(f"Mean CV Accuracy: {cv_scores.mean():.4f}")
print(f"Std Deviation:    {cv_scores.std():.4f}")

if cv_scores.std() < 0.02:
    print("Model is stable across different folds")
else:
    print("High variance across folds - model may be unstable")

In [ ]:
print("\n=== Per-Digit Accuracy ===")
for digit in range(10):
    mask = test_labels == digit
    digit_accuracy = accuracy_score(test_labels[mask], y_pred[mask])
    count = mask.sum()
    print(f"Digit {digit}: {digit_accuracy:.4f} (n={count})")

In [ ]:
import pynvml
import threading
import time

# ── GPU power sampler (runs in background thread) ─────────
power_readings = []
stop_flag = threading.Event()

def sample_gpu_power(interval=0.1):
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)  # GPU 0
    while not stop_flag.is_set():
        mw = pynvml.nvmlDeviceGetPowerUsage(handle)  # milliwatts
        power_readings.append(mw / 1000)             # convert to watts
        time.sleep(interval)
    pynvml.nvmlShutdown()

# ── Measure during inference ──────────────────────────────
sampler = threading.Thread(target=sample_gpu_power)
sampler.start()

infer_start = time.time()
y_pred = model.predict(test_images_reshaped)
infer_end = time.time()

stop_flag.set()
sampler.join()

inference_time = infer_end - infer_start
avg_power_w    = sum(power_readings) / len(power_readings)
energy_j       = avg_power_w * inference_time          # Joules = Watts × seconds
energy_per_img = (energy_j / len(test_images_reshaped)) * 1000  # mJ per image

print(f"Avg GPU power:         {avg_power_w:.2f} W")
print(f"Total energy:          {energy_j:.4f} J")
print(f"Energy per image:      {energy_per_img:.4f} mJ")